# 🚀 Hệ thống Nhận diện Rác 2 Giai đoạn (SAHI + Tiled Training)

Notebook này thực thi **TOÀN BỘ** quy trình nâng cao từ đầu đến cuối trên Kaggle.

### 📐 Sự khác biệt so với Pipeline thường:
1. **Tiling:** Cắt bộ dữ liệu ảnh 4K gốc thành các ảnh lưới 640x640.
2. **Stage 1 (Phát hiện):** YOLO26s được huấn luyện trực tiếp trên tập dữ liệu cắt lưới.
3. **Stage 2 (Phân loại):** EfficientNet-B2.
4. **Inference (SAHI):** Cắt ảnh lúc test để chạy YOLO, sau đó gộp Bounding Box lại (NMS) và đưa vào Stage 2 phân loại.


In [ ]:
# ============================================================
# 1. Tải Mã nguồn & Cài đặt thư viện
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q sahi ultralytics timm


In [ ]:
# ============================================================
# 2. Nhập Dữ liệu Ngoại lai (TACO, TrashNet, RealWaste)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Đã tải: {os.path.basename(task['src'])}")
    else:
        print(f"Bỏ qua: {task['src']} (Không tìm thấy trên Kaggle Dataset)")

In [ ]:
# ============================================================
# 3. Chạy Toàn bộ Tiền xử lý Dữ liệu (Data Pipeline)
# ============================================================
%cd /kaggle/working/waste-detection2-Stage

print("\n--- 3.1 Dọn dẹp & Tạo nhãn YOLO ---")
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

print("\n--- 3.2 Chuẩn bị dữ liệu cho Stage 2 (Classifier) ---")
!python src/data_prep/crop_for_classification.py
!python src/data_prep/merge_external_datasets.py
!python src/data_prep/generate_background.py

print("\n--- 3.3 Tiling dữ liệu Binary cho Stage 1 (Tối ưu cho SAHI) ---")
!python src/data_prep/tiling_binary.py

print("\n✅ Hoàn tất chuẩn bị 100% dữ liệu!")


In [ ]:
# ============================================================
# 4. Huấn luyện Stage 1 (YOLO26s) trên dữ liệu Tiled
# ============================================================
from ultralytics import YOLO
import time
from pathlib import Path

DATASET_YAML = "/kaggle/working/waste-detection2-Stage/data/processed_binary_tiled/dataset.yaml"
OUTPUT_DIR   = "/kaggle/working/waste-detection2-Stage/results/yolo26s_tiled_runs"

print("🚀 Bắt đầu train Stage 1 trên dữ liệu Tiled (YOLO26s – 100 epochs)")
model_s = YOLO("yolo26s.pt")
t0 = time.time()
model_s.train(
    data=DATASET_YAML, imgsz=640, epochs=100, batch=16, patience=20,
    optimizer="auto", lr0=0.01, cos_lr=True, augment=True, workers=4,
    project=OUTPUT_DIR, name="stage1_tiled", exist_ok=True, save=True, save_period=-1,
)
print(f"\n✅ Train Stage 1 hoàn tất sau {(time.time()-t0)/60:.1f} phút")

!cp results/yolo26s_tiled_runs/stage1_tiled/weights/best.pt stage1_tiled_best.pt


In [ ]:
# ============================================================
# 5. Huấn luyện Stage 2 (EfficientNet-B2 – 6 Lớp Classifier)
# Dropout=0.5 | WeightDecay=1e-4 | Mixup(alpha=0.3) | TTA
# ============================================================

!python src/train_stage2_classifier.py

In [ ]:
# ============================================================
# 6. Đánh giá Toàn Trình bằng thuật toán SAHI
# ============================================================
import subprocess, sys, shutil
from pathlib import Path

# Bước 6a: Lọc ra 48 ảnh test chuẩn (có cả nhãn binary và multi-class)
img_binary = Path("data/processed_binary/images/test")
lbl_multi  = Path("data/processed/labels/test")

tmp_img = Path("/kaggle/working/tmp_eval_imgs_strict")
tmp_lbl = Path("/kaggle/working/tmp_eval_lbls_strict")

for d in [tmp_img, tmp_lbl]:
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True)

valid_pairs = []
for img_path in sorted(img_binary.rglob("*")):
    if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'): continue
    rel_path = img_path.relative_to(img_binary)
    multi_lbl_path = lbl_multi / rel_path.with_suffix('.txt')
    if multi_lbl_path.exists():
        valid_pairs.append((img_path, multi_lbl_path))

for img, lbl in valid_pairs:
    flat_name = "__".join(img.relative_to(img_binary).parts)
    shutil.copy2(img, tmp_img / flat_name)
    shutil.copy2(lbl, tmp_lbl / Path(flat_name).with_suffix('.txt').name)

print(f"[INFO] Đã chuẩn bị {len(valid_pairs)} ảnh test.")

# Bước 6b: Chạy đánh giá SAHI
print("[INFO] Đang chạy SAHI inference (slice=512, overlap=0.2)...")
subprocess.run([
    sys.executable, "src/evaluate_2stage_sahi.py",
    "--detector",   "stage1_tiled_best.pt",
    "--classifier", "models/stage2_best.pth",
    "--data-dir",   str(tmp_img),
    "--label-dir",  str(tmp_lbl),
    "--conf",       "0.35",
    "--slice-size", "512",
    "--overlap",    "0.2"
])
print("\n✅ ĐÁNH GIÁ HOÀN TẤT!")


In [ ]:
# ============================================================
# 7. Zip tất cả kết quả để tải về
# ============================================================
import shutil, os

shutil.make_archive("/kaggle/working/final_results", "zip", "/kaggle/working/waste-detection2-Stage/results")
print("\n📦 Đã nén toàn bộ logs, biểu đồ và kết quả: /kaggle/working/final_results.zip")

if os.path.exists("/kaggle/working/waste-detection2-Stage/models"):
    shutil.make_archive("/kaggle/working/final_models", "zip", "/kaggle/working/waste-detection2-Stage/models")
    print("📦 Đã nén trọng số mô hình: /kaggle/working/final_models.zip")
